# CausaSent ABSA — Kaggle Training Notebook

Trains PhoBERT-large with two heads (ATE + binary sentiment) on the CausaSent ABSA dataset.

## Setup required before running
1. Add the processed data as a Kaggle dataset: upload `data/processed/train.jsonl`, `val.jsonl`, `test.jsonl`  
   → Dataset name: `causasent-processed`
2. Enable GPU (T4 x2 recommended) in Settings → Accelerator
3. Run all cells top-to-bottom

In [ ]:
# Install dependencies
!pip install -q transformers==4.44.2 py_vncorenlp pydantic
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
import os, sys, json, random
from pathlib import Path
from collections import defaultdict, Counter
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_DIR = Path('/kaggle/input/causasent-processed')
OUT_DIR = Path('/kaggle/working')
CKPT_DIR = OUT_DIR / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# Reproducibility
SEED = 42
random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Data dir:', DATA_DIR)
print('Files:', list(DATA_DIR.glob('*.jsonl')))

In [ ]:
# ── Label schema ──────────────────────────────────────────────────────────────
ASPECTS = (
    'delivery', 'packaging', 'product_quality',
    'price', 'customer_service', 'usability', 'appearance'
)

ATE_LABELS = ['O'] + [f'B-{a}' for a in ASPECTS] + [f'I-{a}' for a in ASPECTS]
ATE_LABEL2ID = {l: i for i, l in enumerate(ATE_LABELS)}
ATE_ID2LABEL = {i: l for l, i in ATE_LABEL2ID.items()}

SENTIMENT_LABELS = ['positive', 'negative']
SENTIMENT_LABEL2ID = {l: i for i, l in enumerate(SENTIMENT_LABELS)}
SENTIMENT_ID2LABEL = {i: l for l, i in SENTIMENT_LABEL2ID.items()}

IGNORE_INDEX = -100

print(f'ATE labels ({len(ATE_LABELS)}):', ATE_LABELS)
print(f'Sentiment labels:', SENTIMENT_LABELS)

In [ ]:
# ── VnCoreNLP setup ───────────────────────────────────────────────────────────
import py_vncorenlp

VNCORENLP_DIR = str(OUT_DIR / 'vncorenlp')
os.makedirs(VNCORENLP_DIR, exist_ok=True)

prev_cwd = os.getcwd()
if not any(Path(VNCORENLP_DIR).glob('VnCoreNLP-*.jar')):
    print('Downloading VnCoreNLP...')
    py_vncorenlp.download_model(save_dir=VNCORENLP_DIR)

SEGMENTER = py_vncorenlp.VnCoreNLP(annotators=['wseg'], save_dir=VNCORENLP_DIR)
os.chdir(prev_cwd)
print('VnCoreNLP ready')

def segment(text: str) -> list:
    sentences = SEGMENTER.word_segment(text)
    return [w for s in sentences for w in s.split()]

In [ ]:
# ── Span alignment helpers ────────────────────────────────────────────────────
import re

def word_boundaries(text, words):
    spans = []
    cursor = 0
    for w in words:
        surface = w.replace('_', ' ')
        while cursor < len(text) and text[cursor].isspace():
            cursor += 1
        idx = text.find(surface, cursor)
        if idx < 0:
            pat = re.escape(surface).replace(r'\ ', r'\s+')
            m = re.search(pat, text[cursor:])
            if not m:
                continue
            idx = cursor + m.start()
            end = cursor + m.end()
        else:
            end = idx + len(surface)
        spans.append((idx, end))
        cursor = end
    return spans


def expand_to_word_boundary(span, word_spans):
    s, e = span
    ns, ne = s, e
    for ws, we in word_spans:
        if ws <= s < we:
            ns = min(ns, ws)
        if ws < e <= we:
            ne = max(ne, we)
    return ns, ne


def words_in_span(span, word_spans):
    s, e = span
    return [i for i, (ws, we) in enumerate(word_spans) if ws < e and we > s]

In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
@dataclass
class TaggedExample:
    review_id: str
    words: list
    ate_labels: list   # per word
    sent_labels: list  # per word (IGNORE_INDEX except at B-token positions)


def load_absa_jsonl(path):
    groups = {}
    order = []
    for line in Path(path).read_text('utf-8').splitlines():
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        rid = rec['id']
        if rid not in groups:
            groups[rid] = {'id': rid, 'review': rec['review'], 'annotations': []}
            order.append(rid)
        groups[rid]['annotations'].append({
            'aspect_term_span': rec['aspect_term_span'],
            'aspect_category':  rec.get('aspect_category', rec.get('aspect', '')),
            'sentiment':        rec['sentiment'],
        })
    return [groups[r] for r in order]


def build_tagged_examples(reviews):
    out = []
    for r in reviews:
        words = segment(r['review'])
        if not words:
            continue
        n = len(words)
        w_spans = word_boundaries(r['review'], words)
        n = min(n, len(w_spans))
        ate = [ATE_LABEL2ID['O']] * n
        sent = [IGNORE_INDEX] * n

        for ann in r['annotations']:
            asp = ann['aspect_category']
            sentiment = ann['sentiment']
            b_tag = f'B-{asp}'
            if b_tag not in ATE_LABEL2ID or sentiment not in SENTIMENT_LABEL2ID:
                continue
            sc, ec = ann['aspect_term_span']
            snapped = expand_to_word_boundary((sc, ec), w_spans)
            widx = words_in_span(snapped, w_spans)
            if not widx:
                continue
            for k, wi in enumerate(widx):
                if wi >= n:
                    continue
                pos = 'B' if k == 0 else 'I'
                ate[wi] = ATE_LABEL2ID[f'{pos}-{asp}']
                if k == 0:
                    sent[wi] = SENTIMENT_LABEL2ID[sentiment]
        out.append(TaggedExample(r['id'], words, ate, sent))
    return out


def word_ids_slow(tokenizer, words, max_len):
    out = [None]  # CLS
    cls = tokenizer.cls_token_id
    sep = tokenizer.sep_token_id
    n_special = int(cls is not None) + int(sep is not None)
    budget = max_len - n_special
    for wi, word in enumerate(words):
        toks = tokenizer.encode(word, add_special_tokens=False)
        if not toks:
            continue
        if len(out) - 1 + len(toks) > budget:
            break
        out.extend([wi] * len(toks))
    out.append(None)  # SEP
    while len(out) < max_len:
        out.append(None)
    return out[:max_len]


class ABSADataset(Dataset):
    def __init__(self, examples, tokenizer, max_len=128):
        self.examples = examples
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        enc = self.tokenizer(
            ex.words, is_split_into_words=True,
            truncation=True, max_length=self.max_len,
            padding='max_length', return_tensors='pt',
        )
        try:
            wids = enc.word_ids(batch_index=0)
        except Exception:
            wids = word_ids_slow(self.tokenizer, ex.words, self.max_len)
        ate_lbl = [IGNORE_INDEX] * len(wids)
        snt_lbl = [IGNORE_INDEX] * len(wids)
        seen = set()
        for i, wid in enumerate(wids):
            if wid is None or wid in seen:
                continue
            seen.add(wid)
            if wid < len(ex.ate_labels):
                ate_lbl[i] = ex.ate_labels[wid]
                snt_lbl[i] = ex.sent_labels[wid]
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'ate_labels':     torch.tensor(ate_lbl, dtype=torch.long),
            'sent_labels':    torch.tensor(snt_lbl, dtype=torch.long),
        }

In [ ]:
# ── Load and segment data (takes ~5-10 min for full dataset) ──────────────────
print('Loading train...')
train_reviews = load_absa_jsonl(DATA_DIR / 'train.jsonl')
print(f'  {len(train_reviews)} reviews')
train_examples = build_tagged_examples(train_reviews)
print(f'  {len(train_examples)} tagged examples')

print('Loading val...')
val_reviews = load_absa_jsonl(DATA_DIR / 'val.jsonl')
val_examples = build_tagged_examples(val_reviews)
print(f'  {len(val_examples)} tagged examples')

# Aspect distribution check
from collections import Counter
asp_cnt = Counter()
for r in train_reviews:
    for ann in r['annotations']:
        asp_cnt[ann['aspect_category']] += 1
print('\nTrain aspect distribution:')
for asp, n in sorted(asp_cnt.items(), key=lambda x: -x[1]):
    print(f'  {asp:<20} {n}')

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
PRETRAINED = 'vinai/phobert-large'

class PhoBertABSA(nn.Module):
    def __init__(self, pretrained=PRETRAINED, dropout=0.1,
                 ate_w=None, sent_w=None, sent_loss_weight=1.0):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(pretrained)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.ate_head  = nn.Linear(hidden, len(ATE_LABELS))
        self.sent_head = nn.Linear(hidden, len(SENTIMENT_LABELS))
        self.sent_loss_weight = sent_loss_weight
        self.ate_loss_fn  = nn.CrossEntropyLoss(weight=ate_w,  ignore_index=IGNORE_INDEX)
        self.sent_loss_fn = nn.CrossEntropyLoss(weight=sent_w, ignore_index=IGNORE_INDEX)

    def forward(self, input_ids, attention_mask, ate_labels=None, sent_labels=None):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        h = self.dropout(out.last_hidden_state)
        ate_logits  = self.ate_head(h)
        sent_logits = self.sent_head(h)
        loss = None
        if ate_labels is not None:
            ate_loss  = self.ate_loss_fn(ate_logits.view(-1, len(ATE_LABELS)), ate_labels.view(-1))
            sent_loss = self.sent_loss_fn(sent_logits.view(-1, len(SENTIMENT_LABELS)), sent_labels.view(-1))
            loss = ate_loss + self.sent_loss_weight * sent_loss
        return loss, ate_logits, sent_logits

print('Model class defined')

In [ ]:
# ── Training config ───────────────────────────────────────────────────────────
CFG = dict(
    pretrained   = PRETRAINED,
    dropout      = 0.1,
    lr           = 2e-5,
    batch_size   = 16,
    epochs       = 10,
    weight_decay = 0.01,
    warmup_ratio = 0.1,
    grad_clip    = 1.0,
    sent_loss_weight = 1.0,
    max_len      = 128,
)

tokenizer = AutoTokenizer.from_pretrained(PRETRAINED, use_fast=True)

train_ds = ABSADataset(train_examples, tokenizer, CFG['max_len'])
val_ds   = ABSADataset(val_examples,   tokenizer, CFG['max_len'])
train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False, num_workers=2)

# Inverse-frequency class weights for ATE head (O dominates)
ate_counts = Counter()
for i in range(len(train_ds)):
    for l in train_ds[i]['ate_labels'].tolist():
        if l != IGNORE_INDEX:
            ate_counts[l] += 1
total = sum(ate_counts.values()) or 1
ate_w = torch.ones(len(ATE_LABELS))
for c in range(len(ATE_LABELS)):
    ate_w[c] = total / (len(ATE_LABELS) * (ate_counts.get(c, 0) + 1))
ate_w = ate_w.to(DEVICE)

model = PhoBertABSA(
    pretrained=CFG['pretrained'], dropout=CFG['dropout'],
    ate_w=ate_w, sent_w=None, sent_loss_weight=CFG['sent_loss_weight']
).to(DEVICE)

print(f'Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')
print(f'Train batches: {len(train_loader)}  Val batches: {len(val_loader)}')

In [ ]:
# ── Evaluation ────────────────────────────────────────────────────────────────
def bio_segments(tags):
    segs = []
    i, n = 0, len(tags)
    while i < n:
        t = tags[i]
        if t == 'O' or not t.startswith('B-'):
            i += 1; continue
        base = t[2:]
        j = i + 1
        while j < n and tags[j] == f'I-{base}':
            j += 1
        segs.append((i, j, base))
        i = j
    return segs

def entity_f1(pred, gold):
    tp = len(pred & gold)
    p = tp / max(1, len(pred))
    r = tp / max(1, len(gold))
    return 2*p*r / max(1e-9, p+r)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    ate_pred, ate_gold = set(), set()
    sp_correct, sp_total = 0, 0
    doc = 0
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        _, alog, slog = model(batch['input_ids'], batch['attention_mask'])
        ap = alog.argmax(-1).cpu().tolist()
        sp = slog.argmax(-1).cpu().tolist()
        ag = batch['ate_labels'].cpu().tolist()
        sg = batch['sent_labels'].cpu().tolist()
        for j in range(len(ap)):
            valid = [i for i, g in enumerate(ag[j]) if g != IGNORE_INDEX]
            pt = [ATE_ID2LABEL.get(ap[j][i], 'O') for i in valid]
            gt = [ATE_ID2LABEL.get(ag[j][i], 'O') for i in valid]
            for s, e, b in bio_segments(pt): ate_pred.add((doc, s, e, b))
            for s, e, b in bio_segments(gt): ate_gold.add((doc, s, e, b))
            for pi, gi in zip(sp[j], sg[j]):
                if gi == IGNORE_INDEX: continue
                sp_correct += int(pi == gi)
                sp_total += 1
            doc += 1
    ate_f1  = entity_f1(ate_pred, ate_gold)
    sent_acc = sp_correct / max(1, sp_total)
    return ate_f1, sent_acc, (ate_f1 + sent_acc) / 2

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
epochs = CFG['epochs']
total_steps = len(train_loader) * epochs
optim = AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
sched = get_linear_schedule_with_warmup(
    optim,
    num_warmup_steps=int(total_steps * CFG['warmup_ratio']),
    num_training_steps=total_steps,
)

best_metric, history = -1.0, []

for epoch in range(1, epochs + 1):
    model.train()
    running = 0.0
    for step, batch in enumerate(train_loader, 1):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        loss, _, _ = model(**batch)
        optim.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
        optim.step(); sched.step()
        running += loss.item()
        if step % 50 == 0:
            print(f'  epoch {epoch} step {step}/{len(train_loader)} loss={running/step:.4f}')

    # val
    val_loss = 0.0
    with torch.no_grad():
        model.eval()
        for batch in val_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            l, _, _ = model(**batch); val_loss += l.item()
    val_loss /= max(1, len(val_loader))

    ate_f1, sent_acc, mean_m = evaluate(model, val_loader)
    row = dict(epoch=epoch, val_loss=val_loss, ate_f1=ate_f1, sent_acc=sent_acc, mean=mean_m)
    history.append(row)
    print(f'[epoch {epoch}] val_loss={val_loss:.4f} ate_f1={ate_f1:.4f} '
          f'sent_acc={sent_acc:.4f} mean={mean_m:.4f}')

    if mean_m > best_metric:
        best_metric = mean_m
        torch.save({'model': model.state_dict(), 'config': CFG, 'mean_metric': mean_m},
                   CKPT_DIR / 'best.pt')
        print('  ↳ saved best checkpoint')

torch.save({'model': model.state_dict(), 'config': CFG}, CKPT_DIR / 'last.pt')
print(f'\nDone. Best mean_metric={best_metric:.4f}')

In [ ]:
# ── Training history ──────────────────────────────────────────────────────────
import pandas as pd
pd.DataFrame(history).set_index('epoch').round(4)

In [ ]:
# ── Test-set evaluation ───────────────────────────────────────────────────────
test_reviews  = load_absa_jsonl(DATA_DIR / 'test.jsonl')
test_examples = build_tagged_examples(test_reviews)
test_ds = ABSADataset(test_examples, tokenizer, CFG['max_len'])
test_loader = DataLoader(test_ds, batch_size=CFG['batch_size'], shuffle=False, num_workers=2)

# Load best checkpoint
ckpt = torch.load(CKPT_DIR / 'best.pt', map_location=DEVICE)
model.load_state_dict(ckpt['model'])

ate_f1, sent_acc, mean_m = evaluate(model, test_loader)
print(f'TEST  ate_f1={ate_f1:.4f}  sent_acc={sent_acc:.4f}  mean={mean_m:.4f}')

In [ ]:
# ── Save results for download ─────────────────────────────────────────────────
results = {
    'config': CFG,
    'history': history,
    'test': {'ate_f1': ate_f1, 'sent_acc': sent_acc, 'mean_metric': mean_m},
}
(OUT_DIR / 'results.json').write_text(json.dumps(results, indent=2), encoding='utf-8')
print('Saved results.json')
print('Checkpoint saved at:', CKPT_DIR / 'best.pt')